# Data Cleansing

In [1]:
import pandas as pd
from ydata_profiling import ProfileReport
import warnings
warnings.filterwarnings('ignore')

df=pd.read_excel('../data/raw/03_hand_AI_cleaned_data.xlsx')

## 1. Original Data Checking

In [2]:
print('=========================================')
print('1. check original data')
print('=========================================\n')
print('\n✅ shape\n')
print(df.shape)
print('\n✅ columns list\n')
print(df.columns.to_list())
print('\n✅ info statistic\n')
print(df.info())
print('\n✅ basic calculation\n')
print(df.describe())
print('\n✅ sample display\n')
print(df.head(5))

1. check original data


✅ shape

(265, 18)

✅ columns list

['year', 'brand_category', 'brand', 'model', 'segment', 'powertrain', 'battery_capacity', 'range_CLTC', 'length_mm', 'width_mm', 'depth_mm', 'wheel_base', 'body_structure', 'battery_type', 'battery_supplier', 'radar_type', 'ADAS_Level', 'price']

✅ info statistic

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 265 entries, 0 to 264
Data columns (total 18 columns):
 #   Column            Non-Null Count  Dtype  
---  ------            --------------  -----  
 0   year              262 non-null    float64
 1   brand_category    265 non-null    object 
 2   brand             265 non-null    object 
 3   model             264 non-null    object 
 4   segment           265 non-null    object 
 5   powertrain        265 non-null    object 
 6   battery_capacity  252 non-null    object 
 7   range_CLTC        251 non-null    object 
 8   length_mm         265 non-null    int64  
 9   width_mm          265 non-null    object 
 10  

## 2. Duplicated Droping

In [3]:
print('=========================================')
print('2. check duplicated of model')
print('=========================================\n')
print('\n✅ model duplicated')
df.duplicated(subset='model').value_counts()

2. check duplicated of model


✅ model duplicated


False    244
True      21
Name: count, dtype: int64

In [4]:
df.drop_duplicates(subset='model',keep='last',inplace=True)

## 3. Values Converting

In [5]:
print('=========================================')
print('3. unique value of columns')
print('=========================================\n')
for col in df.columns:
    print(f'\n✅ {col}\n')
    print(df[f'{col}'].unique())

3. unique value of columns


✅ year

[2021.   nan 2020. 2022. 2023. 2024. 2025. 2018. 2017. 2026.]

✅ brand_category

['合资品牌' '传统自主母品牌' '造车新势力' '自主高端品牌' '外资/进口' '自主新能源独立子品牌']

✅ brand

['大众' '荣威' '本田' '思皓' '高合' '哪吒' '极氪' '奔驰' '丰田' '威马' '宝马' '几何' '岚图' '红旗'
 '爱驰' '飞凡' '广汽埃安' '长安' '华晨新日' '比亚迪' '小鹏' '天际' '沃尔沃' '福特' '特斯拉' '吉利' '零跑'
 'Jeep' '欧拉' '林肯' '路虎' '雷丁' '朋克' '领克' '马自达' '理想' '奇瑞' '雪佛兰' '日产' '上汽大通'
 '凯迪拉克' '蔚来' '沙龙' '起亚' '奥迪' '魏牌' '广汽丰田' '问界' '极狐' '凌宝' 'smart' '哈弗' '深蓝'
 '阿维塔' '腾势' '创维' '海马' '自游家' '雷达' '捷尼赛思' '合创' '睿蓝' '斯威' '江南' '云度' '五菱'
 '舒享家' '智己' '方程豹' '远航' '东风' '仰望' '小米' '智界' 'Leapmotor' '乐道']

✅ model

['大众ID.4 Crozz' '荣威MarvelR' '本田CR-V' '思皓E10X' '高合Hiphi X' '哪吒U Pro'
 '极氪001' '奔驰E级' '丰田RAV4' '威马W6' '宝马ix3' '威兰达' '几何A Pro' '岚图FREE' '宝马iX'
 '红旗E-QM5' '爱驰U6' '大众ID.6 X' 'R汽车新动版ER6' 'AION S PLUS' '逸动460换电' '华晨新日i03'
 '华晨新日i03A' '比亚迪元Pro' '秦PLUS' '小鹏G3i' '天际ME5' '大众ID.6 Crozz' 'XC40' '比亚迪汉'
 'Mustang Mach-E' 'Model Y' 'ER6' '帝豪EC7' '比亚迪海豚' 'T03' '大指挥官' '欧拉好猫GT木兰'
 '冒险家' '路虎揽胜极光L P300e

### Numerical Type

In [6]:
def extract_average(value):
    if pd.isna(value):
        return None
    value_str = str(value)
    
    # Handle range with "-" (e.g., "55.9-66.2")
    if '-' in value_str:
        parts = value_str.split('-')
        low = float(parts[0])
        high = float(parts[1])
        return (low + high) / 2
    
    # Handle range with "/" (e.g., "442/601")
    if '/' in value_str:
        parts = value_str.split('/')
        nums = [float(p) for p in parts]
        return sum(nums) / len(nums)
    
    # Already a single number
    try:
        return float(value_str)
    except:
        return None

# Apply to three columns
df['battery_capacity_avg'] = df['battery_capacity'].apply(extract_average)
df['range_CLTC_avg'] = df['range_CLTC'].apply(extract_average)
df['price_avg'] = df['price'].apply(extract_average)

### Object Type

In [7]:
#  BODY STRUCTURE: Extract number of seats only
df['seats'] = df['body_structure'].str.extract(r'(\d+)\s*座').astype(float)


#  BATTERY TYPE: Classify into 三元, 磷酸铁锂, BOTH, Unknown
def classify_battery_type(val):
    if pd.isna(val):
        return 'Unknown'
    val_str = str(val)
    has_ncm = '三元' in val_str
    has_lfp = '磷酸铁锂' in val_str
    
    if has_ncm and has_lfp:
        return 'BOTH'
    elif has_ncm:
        return 'NCM'
    elif has_lfp:
        return 'LFP'
    else:
        return 'Other'
df['battery_type_clean'] = df['battery_type'].apply(classify_battery_type)

#  BATTERY SUPPLIER: Extract brand only
def classify_supplier(val):
    if pd.isna(val):
        return 'Unknown'
    val_str = str(val)
    # Contains '时代' or 'CATL' or '宁德'
    if '时代' in val_str or 'CATL' in val_str or '宁德' in val_str:
        return 'CATL'
    # Contains '比亚迪' or '弗迪'
    if '比亚迪' in val_str or '弗迪' in val_str or '众迪'in val_str:
        return 'BYD'
    return 'Other'

df['battery_supplier_clean'] = df['battery_supplier'].apply(classify_supplier)

# RADAR TYPE: Has LiDAR or not (keep missing as NaN)
df['has_lidar'] = df['radar_type'].apply(
    lambda x: 'Yes' if pd.notna(x) and '激光' in str(x) else ('No' if pd.notna(x) else None)
)


## 4. Null Handling

In [8]:
# Calculate missing percentage for each column
missing_percentage = (df.isnull().sum() / len(df)) * 100
missing_percentage = missing_percentage.sort_values(ascending=False)

print('=========================================')
print('4. check null ratio ')
print('=========================================\n')
print('\n✅ column & missing_% \n')
print(missing_percentage)

4. check null ratio 


✅ column & missing_% 

has_lidar                 18.442623
radar_type                18.442623
ADAS_Level                14.344262
range_CLTC_avg             4.918033
battery_capacity_avg       4.918033
battery_capacity           4.918033
range_CLTC                 4.918033
battery_type               1.639344
wheel_base                 0.819672
year                       0.819672
model                      0.409836
depth_mm                   0.000000
width_mm                   0.000000
brand_category             0.000000
length_mm                  0.000000
battery_supplier           0.000000
powertrain                 0.000000
segment                    0.000000
price                      0.000000
brand                      0.000000
price_avg                  0.000000
seats                      0.000000
battery_type_clean         0.000000
battery_supplier_clean     0.000000
body_structure             0.000000
dtype: float64


In [9]:
# if model is null ,just drop the row
df.dropna(subset='model',inplace=True)

### wheel_base

In [10]:
# Calculate correlation
correlation = df['wheel_base'].corr(df['length_mm'])
print(f"Correlation between wheel_base and length_mm: {correlation:.3f}")

Correlation between wheel_base and length_mm: 0.930


In [11]:
# If correlation is strong (> 0.7), fill using length_mm via linear regression
# If correlation is weak (< 0.7), fill with median by segment

if correlation > 0.7:
    from sklearn.linear_model import LinearRegression
    
    # Train on non-missing data
    train = df.dropna(subset=['wheel_base', 'length_mm'])
    model = LinearRegression()
    model.fit(train[['length_mm']], train['wheel_base'])
    
    # Predict missing values
    missing_mask = df['wheel_base'].isna()
    df.loc[missing_mask, 'wheel_base'] = model.predict(df.loc[missing_mask, ['length_mm']])
else:
    # Fill with median by segment
    df['wheel_base'] = df.groupby('segment')['wheel_base'].transform(lambda x: x.fillna(x.median()))

### year & battery_type    

In [12]:
df['year']=df['year'].fillna(df['year'].median())
df['battery_type_clean']=df['battery_type_clean'].fillna('unknown')

### battery_capacity_avg 

In [13]:
df['battery_capacity_avg'] = df.groupby('segment')['battery_capacity_avg'].transform(
    lambda x: x.fillna(x.median()))

### range_CLTC_avg           

In [14]:
# Calculate correlation
correlation = df['range_CLTC_avg'].corr(df['battery_capacity_avg'])
print(f"Correlation between range and battery_capacity: {correlation:.3f}")

Correlation between range and battery_capacity: 0.740


In [15]:
# If correlation is strong (> 0.7), fill using length_mm via linear regression
# If correlation is weak (< 0.7), fill with median by segment

if correlation > 0.7:
    from sklearn.linear_model import LinearRegression
    
    # Train on non-missing data
    train = df.dropna(subset=['range_CLTC_avg', 'battery_capacity_avg'])
    model = LinearRegression()
    model.fit(train[['battery_capacity_avg']], train['range_CLTC_avg'])
    
    # Predict missing values
    missing_mask = df['range_CLTC_avg'].isna()
    df.loc[missing_mask, 'range_CLTC_avg'] = model.predict(df.loc[missing_mask, ['battery_capacity_avg']])
else:
    # Fill with median by segment
    df['range_CLTC_avg'] = df.groupby('segment')['battery_capacity_avg'].transform(lambda x: x.fillna(x.median()))

### ADAS_level & radar_type

In [16]:
# Create a contingency table
cross_tab = pd.crosstab(df['has_lidar'], df['ADAS_Level'])

print(cross_tab)

ADAS_Level   L2  L2+
has_lidar           
No          139    2
Yes           0   57


In [17]:
def fill_adas(row):
    if pd.notna(row['ADAS_Level']):
        return row['ADAS_Level']
    
    # Has LiDAR → high ADAS level
    if row['has_lidar'] == 'Yes':
        return 'L2+'
    
    # No LiDAR: check if it's a small/cheap car
    if row['wheel_base'] < 2200 or row['segment']:
        return 'L1'
    
    # Default for modern EVs without LiDAR
    return 'L1'

df['ADAS_Level'] = df.apply(fill_adas, axis=1)

In [18]:
df['has_lidar']=df['has_lidar'].fillna('unknown')

## 5. Saving

In [19]:
df.info()

<class 'pandas.core.frame.DataFrame'>
Index: 243 entries, 1 to 264
Data columns (total 25 columns):
 #   Column                  Non-Null Count  Dtype  
---  ------                  --------------  -----  
 0   year                    243 non-null    float64
 1   brand_category          243 non-null    object 
 2   brand                   243 non-null    object 
 3   model                   243 non-null    object 
 4   segment                 243 non-null    object 
 5   powertrain              243 non-null    object 
 6   battery_capacity        231 non-null    object 
 7   range_CLTC              231 non-null    object 
 8   length_mm               243 non-null    int64  
 9   width_mm                243 non-null    object 
 10  depth_mm                243 non-null    int64  
 11  wheel_base              243 non-null    float64
 12  body_structure          243 non-null    object 
 13  battery_type            239 non-null    object 
 14  battery_supplier        243 non-null    object 

In [20]:
df.to_excel('../data/raw/04_cleaned_data.xlsx',index=True)